# 수정사항입니다 꼭 실행해주세요!

In [1]:
!mv __init__.py /usr/local/lib/python3.6/dist-packages/pop

mv: cannot stat '__init__.py': No such file or directory


In [2]:
!mv Pilot.py /usr/local/lib/python3.6/dist-packages/pop

mv: cannot stat 'Pilot.py': No such file or directory


----
<br><br>

# 0. 기초 테스트
----
모든 테스트는 최소 <a style="color:#FF0000; font-size:1.5em;">5분</a> 이상 진행해야 합니다.<br>
<span style="color:blue">Shift + Enter </span>를 누르면 차례대로 진행합니다.
<br>아래 코드를 실행시키면 연결된 장치가 감지되는지 [O, X]로 출력합니다.

In [2]:
import subprocess as sp, time
from IPython.display import display, Javascript
from ipywidgets import widgets
from pop import Util

# (SMBus, Address, Name)
devs=(('8','68','6축 센서'),
      ('1','5c','휠/카메라 모터 드라이버'),
      ('1','5e','전/후진 모터 드라이버'),
      ('8','0a','오디오 드라이버'))

print("- 감지된 장치 -")
det=True

for line, dev, name in devs:
    try:
        num=str(int(dev,16))
        ret=sp.check_output(['i2cdetect -y -r '+line+' '+num+' '+num+' | grep -oP "UU|'+dev+'"'], shell=True).decode('UTF-8')
        if dev in ret or 'UU' in ret:
            print("[O] "+name)
    except:
        
        det=False
        print("[X] "+name)
            
Camera, Cds, Pilot = (None, None, None)

if not det: 
    print("\n감지되지 않은 장치가 있습니다.\n 확인 후 테스트를 진행해주세요.") 
else:
    from pop import Camera as cam, Cds as cds, Pilot as pilot, LiDAR
    Camera, Cds, Pilot = (cam, cds, pilot)
    
lab=[]

AC=Pilot.AutoCar()
cam=Camera(300,300)
lidar=LiDAR.Rplidar()

- 감지된 장치 -
[O] 6축 센서
[O] 휠/카메라 모터 드라이버
[O] 전/후진 모터 드라이버
[O] 오디오 드라이버


----
<br><br>

# 1. Cds 센서 테스트
----
아래 코드를 실행시키고 Cds센서를 가리거나 빛을 비춰서 값이 변하는 지 확인합니다.
<br>최소 5분 이상, 센서를 가리고 장비를 뒤집거나 여러 곳으로 이동시켜 확인해야합니다.
<br>한 번 실행하면 30초간 측정합니다.

In [ ]:
cds=Cds(7)
lab.append(widgets.Label(value="Cds : 0"))
display(lab[-1])

dtime = time.time()
while time.time()-dtime<30:
    lab[-1].value="Cds : "+str(cds.read())
    time.sleep(0.1)

----
<br><br>
# 2. 6축 센서 테스트
----
아래 코드를 실행시키고 장비를 움직여 값이 변하는 지 확인합니다.
<br>최소 5분 이상, 센서를 가리고 장비를 뒤집거나 여러 곳으로 이동시켜 확인해야합니다.
<br>한 번 실행하면 30초간 측정합니다.

In [4]:
accX=widgets.Label(value="Acc_X : 0")
accY=widgets.Label(value="Acc_Y : 0")
accZ=widgets.Label(value="Acc_Z : 0")
gyroX=widgets.Label(value="Gyro_X : 0")
gyroY=widgets.Label(value="Gyro_Y : 0")
gyroZ=widgets.Label(value="Gyro_Z : 0")

display(accX)
display(accY)
display(accZ)
display(gyroX)
display(gyroY)
display(gyroZ)

lasttime = 0
dtime = time.time()
while time.time()-dtime<30:
    if time.time()-lasttime>1:
        acc=AC.getAccel()
        gyro=AC.getGyro()

        accX.value="Acc_X : "+str(acc['x'])
        accY.value="Acc_Y : "+str(acc['y'])
        accZ.value="Acc_Z : "+str(acc['z'])
        gyroX.value="Gyro_X : "+str(gyro['x'])
        gyroY.value="Gyro_Y : "+str(gyro['y'])
        gyroZ.value="Gyro_Z : "+str(gyro['z'])
        lasttime=time.time()

Label(value='Acc_X : 0')

Label(value='Acc_Y : 0')

Label(value='Acc_Z : 0')

Label(value='Gyro_X : 0')

Label(value='Gyro_Y : 0')

Label(value='Gyro_Z : 0')

KeyboardInterrupt: 

----
<br><br>
# 3. LiDAR 테스트
----
아래 코드를 실행시키면 LiDAR가 돌면서 주변을 스캔해 실시간 2차원 맵이 표시됩니다.
<br>LiDAR가 돌지 않으면 신호나 전원 연결이 잘못되었을 가능성이 있습니다.

In [7]:
lidar.connect()
lidar.startMotor()
lasttime=0
dtime = time.time()
while time.time()-dtime<30:
    if time.time()-lasttime>0.03:
        data=lidar.getMap(size=(300,300))
        Util.imshow("map", data, width=600, height=600)
        lasttime=time.time()
lidar.stopMotor()

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

KeyboardInterrupt: 

In [8]:
lidar.stopMotor()

----
<br><br>
# 4. 서보 모터 정렬
----
아래 코드를 실행시키면 앞 바퀴가 정면으로 정렬되고, 카메라가 중앙 정면으로 정렬됩니다.
<br>바퀴 또는 카메라가 정면이 아니라면 서보 결합부를 분리하여 재조립해 중앙을 맞춰야 합니다.

In [9]:
AC.camTilt(0)
AC.camPan(90)
AC.steering=0

----
<br><br>
# 5. 모터 테스트 
--------
> 바닥에서 띄운 후 실행할 것.

아래 코드를 실행시키면 바퀴는 좌/우, 전/후진으로 움직이고, 카메라가 좌/우, 앞/뒤로 움직입니다.
<br>바퀴 또는 카메라가 잘 움직이는지 확인합니다.
<br>한 번 실행하면 30초간 측정합니다.


In [7]:
dtime = time.time()
lasttime=0
sw=True
while time.time()-dtime<30:
    if time.time()-lasttime>2:
        if sw:
            AC.camTilt(0)
            AC.camPan(10)
            AC.steering=-1
            AC.forward(99)
            sw=False
        else:
            AC.camTilt(90)
            AC.camPan(180)
            AC.steering=1
            AC.backward(99)
            sw=True
        lasttime=time.time()
        
AC.stop()

KeyboardInterrupt: 

In [8]:
AC.stop()

In [10]:
dtime = time.time()
lasttime=0
sw=True
while time.time()-dtime<10:
    if time.time()-lasttime>5:
        if sw:
            AC.forward(99)
            sw=False
        else:
            AC.backward(99)
            sw=True
        lasttime=time.time()
        
AC.stop()

----
<br><br>
# 6. 오디오 테스트 
--------
아래 코드를 실행시키면 30초간 마이크 테스트를 진행합니다.
<br>좌/우 마이크에 소리가 입력되면 [　　　|　　　] 모양으로 생긴 사운드바가 [-=====|=====-] 으로 바뀝니다.
<br><br>
그 다음 코드를 실행시키면 3분 30초간 음원 파일이 재생됩니다.
<br>좌/우 스피커 출력이 정상인지 확인합니다.



In [11]:
!timeout 30s rec test.mp3 

rec WARN alsa: can't encode 0-bit Unknown or not applicable

Input File     : 'default' (alsa)
Channels       : 2
Sample Rate    : 48000
Precision      : 16-bit
Sample Encoding: 16-bit Signed Integer PCM

In:0.00% 00:00:13.48 [00:00:00.00] Out:647k  [      |      ]        Clip:0    ^C
In:0.00% 00:00:13.65 [00:00:00.00] Out:651k  [      |      ]        Clip:0    
Aborted.


In [12]:
!play test.mp3 

play WARN alsa: can't encode 0-bit Unknown or not applicable

test.mp3:

 File Size: 218k      Bit Rate: 128k
  Encoding: MPEG audio    
  Channels: 2 @ 16-bit   
Samplerate: 48000Hz      
Replaygain: off         
  Duration: 00:00:13.61  

In:99.8% 00:00:13.58 [00:00:00.02] Out:652k  [      |      ]        Clip:0    
Done.


In [13]:
!play 1.mp3 

play WARN alsa: can't encode 0-bit Unknown or not applicable

1.mp3:

 File Size: 6.86M     Bit Rate: 320k
  Encoding: MPEG audio    
  Channels: 2 @ 16-bit   
Samplerate: 48000Hz      
Replaygain: off         
  Duration: 00:02:51.43  Title: [Elancia]~Rolancia

In:3.73% 00:00:06.40 [00:02:45.03] Out:307k  [   -==|==-   ]        Clip:0    
Aborted.


---
아래 코드를 실행시키면 녹음된 파일과 음원 파일을 <span style="color:red">삭제</span>합니다.
<br><span style="color:red">정보 유출</span> 및 <span style="color:red">저작권 문제</span>가 발생할 수 있으므로 테스트 후 꼭 실행해주시기 바랍니다.

In [ ]:
!rm test.mp3

----
<br><br>
# 7. LED 테스트 
--------
아래 코드를 실행시키면 차량 전/후방에 탑재된 LED가 빨간색과 초록색을 번갈아가며 깜빡거립니다.
<br>LED가 잘 작동되는지 확인합니다.



In [20]:
LED=Pilot.PWM(1,0x5c)
LED.setFreq(50)

dtime = time.time()
lasttime=0
sw=0
while time.time()-dtime<30:
    if time.time()-lasttime>1:
        if sw==0:
            LED.setDuty(0,99)
            LED.setDuty(1,99)
            LED.setDuty(2,99)
            LED.setDuty(3,99)
        elif sw==2:
            LED.setDuty(4,99)
            LED.setDuty(5,99)
            LED.setDuty(6,99)
            LED.setDuty(7,99)
        else:
            LED.setDuty(0,0)
            LED.setDuty(1,0)
            LED.setDuty(2,0)
            LED.setDuty(3,0)
            LED.setDuty(4,0)
            LED.setDuty(5,0)
            LED.setDuty(6,0)
            LED.setDuty(7,0)
        sw+=1
        sw%=4
        lasttime=time.time()

----
<br><br>
# 8. 카메라 테스트 
--------
아래 코드를 실행시키면 카메라 영상이 출력됩니다.
<br>카메라 영상이 잘 나오는지, 렌즈에 크랙은 없는지 확인합니다.



In [4]:
cam.show()

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

----
<br><br>

# 9. Peripheral 모듈 테스트
----
Peripheral 모듈에는 피에조 부저, 스위치, LED가 있습니다.
<br> 각각의 기능들을 모두 테스트 합니다.


아래 코드를 실행시키면 '나비야' 음악이 나오며 **피에조 부저**가 정상적으로 작동하는지 확인할 수 있습니다.

In [ ]:
import time
import RPi.GPIO as GPIO

GPIO.setwarnings(False)
GPIO.cleanup()
GPIO.setmode(GPIO.BCM)

# pwm0 강제 초기화 (stale 상태 방지)
try:
    with open('/sys/class/pwm/pwmchip4/unexport', 'w') as f:
        f.write('0')
except OSError:
    pass

# ① GPIO.setup(n, OUT)을 no-op으로 패치
#    → GPIO.PWM이 pwm0를 새로 export → 핀 mux가 PWM 모드로 올바르게 전환
_orig_setup = GPIO.setup
def _pwm_safe_setup(channel, direction, *args, **kwargs):
    if direction == GPIO.OUT:
        return  # GPIO.PWM이 직접 처리하도록 스킵
    return _orig_setup(channel, direction, *args, **kwargs)
GPIO.setup = _pwm_safe_setup

from pop import PiezoBuzzer
p = PiezoBuzzer(12)
GPIO.setup = _orig_setup  # ② 복원

butterfly_scale    = [4,4,4, 4,4,4, 4,4,4,4, 4,4,4,  4,4,4,4, 4,4,4, 4,4,4,4, 4,4,4]
butterfly_pitch    = [8,5,5, 6,3,3, 1,3,5,6, 8,8,8,  8,5,5,5, 6,3,3, 1,5,8,8, 5,5,5]
butterfly_duration = [8,8,4, 8,8,4, 8,8,8,8, 8,8,4,  8,8,8,8, 8,8,4, 8,8,8,8, 8,8,4]
sheet_butterfly = [butterfly_scale, butterfly_pitch, butterfly_duration]

p.play(sheet_butterfly)

<br><br>아래 코드를 실행시키면 30초간 **스위치**가 정상적으로 작동하는지 확인할 수 있습니다.

In [7]:
from pop import Input

sw1 = Input(16)
sw2 = Input(13)

SW1=widgets.Label(value="SW1 : 0")
SW2=widgets.Label(value="SW2 : 0")
display(SW1)
display(SW2)

dtime = time.time()
while time.time()-dtime<30:
    SW1.value="SW1 : "+str(sw1.read())
    SW2.value="SW2 : "+str(sw2.read())

Label(value='SW1 : 0')

Label(value='SW2 : 0')

<br><br>아래 코드를 실행시키면 15초간 **LED**가 정상적으로 작동하는지 확인할 수 있습니다.

In [ ]:
from pop import PwmController

pwm = PwmController(0x57)
pwm.init()
pwm.setChannel(-1)
pwm.setFreq(50)
pwm.setDuty(0)

for i in range(30):
    if i % 2:
        pwm.setDuty(0)
    else:
        pwm.setDuty(100)
    time.sleep(0.5)

In [18]:
pwm.setChannel(0)
pwm.setDuty(100)

pwm.setChannel(1)
pwm.setDuty(100)